In [1]:
!pip install pandas openpyxl rapidfuzz

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 7.3 MB/s  0:00:00


In [5]:
import sqlite3
import pandas as pd
import re
from rapidfuzz import process, fuzz

ALDI_DB = "aldi_products.db"
COFID_XLSX = "CoFID_2021.xlsx"

def import_cofid_to_sqlite():
    """
    Dynamically loads CoFID sheets, cleans stacked multi-row headers,
    filters structural clutter, and streams clean data into SQLite.
    """
    print("Parsing CoFID Excel Workbook...")
    
    target_sheet = "1.3 Proximates"
    try:
        # Load the raw sheet entirely without skipping rows to parse structural offsets
        df_raw = pd.read_excel(COFID_XLSX, sheet_name=target_sheet, header=None)
    except Exception:
        target_sheet = "Proximates"
        df_raw = pd.read_excel(COFID_XLSX, sheet_name=target_sheet, header=None)

    # 1. Identify rows dynamically
    header_row_idx = None
    data_start_row_idx = None
    
    # CoFID pattern: '13-145', '14-870' etc. (Two digits, a hyphen, then 3 or more digits)
    food_code_pattern = re.compile(r"^\d{2}-\d{3,}")

    for idx, row in df_raw.iterrows():
        row_values = [str(x).strip() for x in row.values]
        
        # Identify the primary header line containing the code and item name
        if header_row_idx is None:
            if any("food code" in x.lower() for x in row_values) and any("food name" in x.lower() for x in row_values):
                header_row_idx = idx
                continue
                
        # Find where actual data starts by testing column 0 for a real food code index string
        val_first_col = str(row.iloc[0]).strip()
        if food_code_pattern.match(val_first_col):
            data_start_row_idx = idx
            break

    if header_row_idx is None or data_start_row_idx is None:
        raise ValueError("Could not dynamically isolate the multi-row header boundary configurations.")

    print(f"Structure Found -> Headers Row: {header_row_idx} | Data Starts Row: {data_start_row_idx}")

    # 2. Extract column header strings using the identified header row
    headers = [str(col).strip().lower() for col in df_raw.iloc[header_row_idx].values]
    
    # Slice the dataframe to isolate data blocks only (dropping header rows and blank rows)
    df_data = df_raw.iloc[data_start_row_idx:].copy()
    df_data.columns = headers

    # 3. Build a dynamic keyword map to normalize case variations
    column_mapping = {}
    for col in df_data.columns:
        if "food code" in col:
            column_mapping[col] = "food_code"
        elif "food name" in col:
            column_mapping[col] = "food_name"
        elif "protein" in col:
            column_mapping[col] = "protein_g"
        elif "fat" in col and "fatty" not in col:
            column_mapping[col] = "fat_g"
        elif "carbohydrate" in col:
            column_mapping[col] = "carbohydrate_g"
        elif "energy (kcal)" in col:
            column_mapping[col] = "energy_kcal"

    df_data = df_data.rename(columns=column_mapping)
    
    # Keep only the columns required for the optimization matrix
    keep_cols = ["food_code", "food_name", "energy_kcal", "protein_g", "fat_g", "carbohydrate_g"]
    df_clean = df_data[[c for c in df_data.columns if c in keep_cols]].copy()
    
    # Drop any remaining unneeded trailing note rows or empty structural rows
    df_clean = df_clean.dropna(subset=["food_code", "food_name"])
    df_clean = df_clean[df_clean["food_code"].astype(str).str.contains(r"\d{2}-\d{3,}", regex=True)]

    # 4. Convert trace indicators safely
    # CoFID uses text identifiers like 'Tr' (Trace) or 'N' (No entry). Change them to 0.0
    for nutrient in ["energy_kcal", "protein_g", "fat_g", "carbohydrate_g"]:
        if nutrient in df_clean.columns:
            df_clean[nutrient] = df_clean[nutrient].astype(str).str.replace(r'(?i)tr|n', '0', regex=True)
            df_clean[nutrient] = df_clean[nutrient].str.replace(r'[^\d\.]', '', regex=True)
            df_clean[nutrient] = pd.to_numeric(df_clean[nutrient], errors='coerce').fillna(0.0)
        else:
            df_clean[nutrient] = 0.0

    # Ensure clean string data types for strings
    df_clean['food_code'] = df_clean['food_code'].astype(str).str.strip()
    df_clean['food_name'] = df_clean['food_name'].astype(str).str.strip()

    # --- ADD THIS LINE TO FIX THE UNIQUE CONSTRAINT ERROR ---
    # This removes duplicate rows sharing the exact same food_code
    df_clean = df_clean.drop_duplicates(subset=["food_code"], keep="first")

    # 5. Save cleanly into SQLite
    conn = sqlite3.connect(ALDI_DB)
    df_clean.to_sql("nutrients", conn, if_exists="replace", index=False)
    
    cursor = conn.cursor()
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_food_code ON nutrients(food_code);")
    conn.commit()
    conn.close()
    
    print(f"✓ Imported {len(df_clean)} clean nutritional records into the 'nutrients' table!")

def build_linking_table():
    conn = sqlite3.connect(ALDI_DB)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS product_nutrition_matches (
            sku TEXT,
            food_code TEXT,
            match_confidence REAL,
            manual_override INTEGER DEFAULT 0,
            PRIMARY KEY (sku),
            FOREIGN KEY (sku) REFERENCES products(sku),
            FOREIGN KEY (food_code) REFERENCES nutrients(food_code)
        )
    ''')
    conn.commit()
    conn.close()

def auto_match_products():
    print("\nRunning matching engine...")
    conn = sqlite3.connect(ALDI_DB)
    
    aldi_df = pd.read_sql("SELECT sku, name FROM products", conn)
    cofid_df = pd.read_sql("SELECT food_code, food_name FROM nutrients", conn)
    
    if aldi_df.empty or cofid_df.empty:
        print("Verification Error: Empty data arrays discovered.")
        conn.close()
        return

    cofid_choices = list(zip(cofid_df['food_name'], cofid_df['food_code']))
    cofid_names = [x[0] for x in cofid_choices]
    
    matches_to_insert = []
    
    for idx, row in aldi_df.iterrows():
        sku = row['sku']
        aldi_name = row['name']
        if not aldi_name: continue
            
        best_match = process.extractOne(aldi_name, cofid_names, scorer=fuzz.token_sort_ratio)
        if best_match:
            matched_name, confidence, match_idx = best_match
            matched_code = cofid_choices[match_idx][1]
            matches_to_insert.append((sku, matched_code, round(confidence, 2)))

    cursor = conn.cursor()
    cursor.executemany('''
        INSERT OR REPLACE INTO product_nutrition_matches (sku, food_code, match_confidence)
        VALUES (?, ?, ?)
    ''', matches_to_insert)
    
    conn.commit()
    conn.close()
    print(f"✓ Successfully processed data mapping connections for {len(matches_to_insert)} items.")

if __name__ == "__main__":
    import_cofid_to_sqlite()
    build_linking_table()
    auto_match_products()

Parsing CoFID Excel Workbook...
Structure Found -> Headers Row: 0 | Data Starts Row: 3
✓ Imported 2886 clean nutritional records into the 'nutrients' table!

Running matching engine...
✓ Successfully processed data mapping connections for 4820 items.
